<a href="https://colab.research.google.com/github/Nikolai-N484/Data201_NikolaiN/blob/main/Week4/Week_4_Assignment_Nikolai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Logistic Regression: R → Python Bridge**

## **Nikolai Navarro**

---

## Learning Objectives
By the end of this assignment, you should be able to:

- Fit and interpret logistic regression models
- Translate workflows from R (glm / tidymodels) to Python
- Interpret odds ratios
- Evaluate classification models using Accuracy and ROC–AUC
- Reflect on the difference between statistical inference vs prediction

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve

df = pd.read_csv("https://raw.githubusercontent.com/Reben80/Data201/refs/heads/main/Dataset/housing.csv")
df

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,145143.0,1280.741760,1.0,Suburb,Townhouse
1,100002,152251.0,1406.283113,2.0,Uptown,SingleFamily
2,100003,148251.0,4146.825713,6.0,Suburb,MultiFamily
3,100004,177711.0,3946.599818,6.0,Suburb,SingleFamily
4,100005,155269.0,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,232811.0,1443.241197,3.0,Midtown,Condo
596,100597,235624.0,1083.909714,2.0,Suburb,Condo
597,100598,244889.0,1600.126432,1.0,Suburb,SingleFamily
598,100599,239545.0,1248.216637,1.0,Waterfront,Condo


---

## Step 0 - Create a Binary Outcome
For classification, convert price into a binary variable.

```
high_price = price > median(price)
```
This creates two groups:

- `1` → expensive homes
- `0` → less expensive homes

In [4]:
df["high_price"] = (df["price"] > df["price"].median()).astype(int)
df

,listing_id,price,size,bedrooms,neighborhood,type,high_price
0,100001,145143.0,1280.741760,1.0,Suburb,Townhouse,0
1,100002,152251.0,1406.283113,2.0,Uptown,SingleFamily,0
2,100003,148251.0,4146.825713,6.0,Suburb,MultiFamily,0
3,100004,177711.0,3946.599818,6.0,Suburb,SingleFamily,0
4,100005,155269.0,1243.751760,1.0,Downtown,MultiFamily,0
...,...,...,...,...,...,...,...
595,100596,232811.0,1443.241197,3.0,Midtown,Condo,1
596,100597,235624.0,1083.909714,2.0,Suburb,Condo,1
597,100598,244889.0,1600.126432,1.0,Suburb,SingleFamily,1
598,100599,239545.0,1248.216637,1.0,Waterfront,Condo,1


---
## Part A - Logistic Regression for Inference

In Python

Fit the equivalent model using statsmodels.

Example:
```
import statsmodels.formula.api as smf

model = smf.logit(
    "high_price ~ size + bedrooms + C(neighborhood)",
    data=df
).fit()

print(model.summary())
```

Report

Create a table including:

- coefficients
- odds ratios
- p-values

Odds ratios:
```
odds_ratio = exp(coefficient)
```

In [7]:
model = smf.logit(
    "high_price ~ size + bedrooms + C(neighborhood) + C(type)", data=df
).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.683349
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:             high_price   No. Observations:                  524
Model:                          Logit   Df Residuals:                      514
Method:                           MLE   Df Model:                            9
Date:                Sun, 15 Mar 2026   Pseudo R-squ.:                 0.01414
Time:                        04:45:25   Log-Likelihood:                -358.07
converged:                       True   LL-Null:                       -363.21
Covariance Type:            nonrobust   LLR p-value:                    0.3292
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         0.2590      0.309      0.837      0.

In [15]:
coef = np.exp(model.params)
odds_ratio = np.exp(coef)
p_value = model.pvalues
